# Advanced Artificial Intelligence Task 2: 
## Inference Example

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
from pathlib import Path

# Load tuned model weights into new model
MODEL_WEIGHTS = "EfficientNet_V2_S.pth"
MODEL = models.efficientnet_v2_s(weights=None) 
in_features = MODEL.classifier[1].in_features
MODEL.classifier[1] = nn.Linear(in_features, 2)

# Load mmodel weights
MODEL.load_state_dict(torch.load(MODEL_WEIGHTS, weights_only=True))

# Send to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(device)

# Set model to evaluate mode
MODEL.eval() 

# Get transforms for inference input
auto_transforms = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1.transforms()

# Define our classes
class_names = ['Healthy', 'Rotten']

def classify_produce(image_path):
    """
    Classify a single image of produce as Healthy or Rotten.
    Returns the predicted label and confidence percentage.
    """
    # Open image and ensure it's RGB
    image = Image.open(image_path).convert("RGBA").convert("RGB")
    
    # Apply transforms and add a batch dimension (Required)
    # Unsqueeze turns shape [3, 224, 224] into [1, 3, 224, 224]
    input_tensor = auto_transforms(image).unsqueeze(0).to(device)
    
    # Turn off gradients for inference
    with torch.no_grad():
        outputs = MODEL(input_tensor)
        
        # Get the highest probability class
        _, pred_id = torch.max(outputs, 1)
        
        # Optional: Calculate the actual confidence percentage using Softmax
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
        probability = probabilities[pred_id].item() * 100
        
    prediction_label = class_names[pred_id.item()]
    
    print(f"Prediction: {prediction_label}")
    print(f"Confidence: {probability:.2f}%")
    
    return prediction_label, probability



FileNotFoundError: [Errno 2] No such file or directory: 'EfficientNet_V2_S.pth'

In [ ]:
test_image = "download (3).jpg"
image_path = Path(".") / "data" / "test_data" / test_image
classify_produce(image_path)

Prediction: Healthy
Confidence: 98.81%


('Healthy', 98.8052248954773)